In [ ]:
import os, sys
from pathlib import Path


PROJECT_ROOT = Path("/home/mame_hil")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

os.makedirs("output/ana", exist_ok=True)

import pandas as pd
from pathlib import Path
import numpy as np

from abx_app.AttackCNN.utils.condition import Condition

In [ ]:
subjects = [
    "s00",
    "s01",
    "s02",
    "s03",
    "s04",
    "s05",
    "s06",
    "s07",
]
layers = ["conv1", "layer3", "avgpool"]

results_by_subject_layer = {}

for subject in subjects:
    path_csv = Path(f"results/{subject}/summary_thresholds.csv")
    data = pd.read_csv(path_csv)
    data["Condition"] = data["name_cond"].apply(Condition.from_string)  # type: ignore

    for _, row in data.iterrows():
        cond: Condition = row["Condition"]
        key = (subject, cond.layer, cond.component, cond.direction)
        if key not in results_by_subject_layer:
            results_by_subject_layer[key] = {"ecc": [], "mean_threshold": []}
        results_by_subject_layer[key]["ecc"].append(cond.ecc)
        results_by_subject_layer[key]["threshold"].append(row["threshold"])

In [ ]:
# Aggregate data by subject and layer
aggregated_data = {subject: {layer: {} for layer in layers} for subject in subjects}

for (subject, layer, component, direction), values in results_by_subject_layer.items():
    eccs = np.array(values["ecc"])
    thresholds = np.array(values["threshold"])
    for ecc in np.unique(eccs):
        mask = eccs == ecc
        mean_value = thresholds[mask].mean()
        if ecc not in aggregated_data[subject][layer]:
            aggregated_data[subject][layer][ecc] = []
        aggregated_data[subject][layer][ecc].append(mean_value)

In [5]:
# Compute mean and SD across components and directions for each subject
subject_means = {subject: {layer: {} for layer in layers} for subject in subjects}
subject_sds = {subject: {layer: {} for layer in layers} for subject in subjects}

for subject, layer_data in aggregated_data.items():
    for layer, ecc_data in layer_data.items():
        for ecc, values in ecc_data.items():
            subject_means[subject][layer][ecc] = round(np.mean(values), 4)
            subject_sds[subject][layer][ecc] = round(np.std(values), 4)

In [10]:
# Compute overall mean and SD across subjects for each layer and ecc
overall_means = {layer: {ecc: [] for ecc in [4, 8, 12]} for layer in layers}
overall_sds = {layer: {ecc: [] for ecc in [4, 8, 12]} for layer in layers}

for subject, layer_data in subject_means.items():
    for layer, ecc_data in layer_data.items():
        for ecc, mean_value in ecc_data.items():
            overall_means[layer][ecc].append(mean_value)

final_means = {layer: {} for layer in layers}
final_sds = {layer: {} for layer in layers}

for layer, ecc_data in overall_means.items():
    for ecc, values in ecc_data.items():
        final_means[layer][ecc] = round(np.mean(values), 4)
        final_sds[layer][ecc] = round(np.std(values), 4)

print(final_means)
print(final_sds)

{'conv1': {4: 47.915, 8: 63.0404, 12: 63.7918}, 'layer3': {4: 0.486, 8: 0.5983, 12: 0.6407}, 'avgpool': {4: 0.1538, 8: 0.153, 12: 0.1688}}
{'conv1': {4: 8.8166, 8: 11.8613, 12: 8.5024}, 'layer3': {4: 0.1275, 8: 0.0686, 12: 0.0533}, 'avgpool': {4: 0.0182, 8: 0.0253, 12: 0.0203}}


In [7]:
# Prepare table data for each subject and overall summary
table_data = []
for subject in subjects:
    for layer in layers:
        for ecc in [4, 8, 12]:
            mean_value = subject_means[subject][layer].get(ecc, np.nan)
            sd_value = subject_sds[subject][layer].get(ecc, np.nan)
            table_data.append(
                {
                    "Subject": subject,
                    "Layer": layer,
                    "Eccentricity": ecc,
                    "Mean Threshold": mean_value,
                    "SD (Component x Direction)": sd_value,
                }
            )

In [8]:
# Add overall summary to the table
for layer in layers:
    for ecc in [4, 8, 12]:
        overall_mean = final_means[layer].get(ecc, np.nan)
        overall_sd = final_sds[layer].get(ecc, np.nan)
        table_data.append(
            {
                "Subject": "Overall",
                "Layer": layer,
                "Eccentricity": ecc,
                "Mean Threshold": overall_mean,
                "SD (Across Subjects)": overall_sd,
            }
        )

In [9]:
# Convert to DataFrame and save as CSV
table_df = pd.DataFrame(table_data)
output_path = Path("output/ana/subject_layer_thresholds_jupyter.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
table_df.to_csv(output_path, index=False)

print(f"Data saved to {output_path}")

Data saved to output/ana/subject_layer_thresholds_jupyter.csv
